# Face v2 — round 8: fine-tune HSEmotion B2 on RAVDESS 24 actors (leave-actors-out)
Same folds as the face models' comparison (actors[i::6]): 6 folds → out-of-fold outputs for every RAVDESS crop (evaluated on the laptop
with the same samples / metrics: current head 0.723, blendshape model 0.591), then fold "all" → `best.onnx` for the web.
Everything is written to `MyDrive/face-v2/out/rav_ft/` as it finishes, so if Colab disconnects just **Run all** again: finished folds are skipped.

Needs in `MyDrive/face-v2/`: `Face-Lab-code-r8.zip`. Runtime → **T4 GPU**. About 1–1.5 h.

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)
else:
    raise SystemExit('NO GPU: Runtime > Change runtime type > T4 GPU, then Run all again')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, glob, hashlib
D='/content/drive/MyDrive/face-v2'; OUT=f'{D}/out/rav_ft'
os.makedirs(OUT, exist_ok=True); os.makedirs('/content/Main_demo', exist_ok=True)
Z = f'{D}/Face-Lab-code-r8.zip'
if not os.path.exists(Z):                      # uploaded in pieces (r8_parts/r8.zip.p00 …) → join them back
    ps = sorted(glob.glob(f'{D}/r8_parts/r8.zip.p*')); print(len(ps), 'parts')
    Z = '/content/r8.zip'
    with open(Z, 'wb') as o:
        for p in ps: o.write(open(p, 'rb').read())
h = hashlib.sha256(open(Z, 'rb').read()).hexdigest()
assert h == 'd15ba5fcc8072a567729f72324afc4df88001731def6b19387dbd8427a432832', ('zip checksum mismatch', h)
zipfile.ZipFile(Z).extractall('/content/Main_demo')
%cd /content/Main_demo/Face-Lab
!pip -q install onnx onnx2torch onnxruntime
!mkdir -p hse && wget -q -O hse/enet_b2_8.onnx https://github.com/av-savchenko/face-emotion-recognition/raw/main/models/affectnet_emotions/onnx/enet_b2_8.onnx && ls -la hse cache
print(sorted(os.listdir(OUT)))

In [ ]:
!python ft_rav.py --out {OUT} --folds 0,1,2,3,4,5,all --epochs 2 --lr 3e-5 --kd 0.5

In [ ]:
# pack the outputs in parts < 9 MB (so they can be pulled back to the laptop) → out/rav_ft_parts/
import glob
P = f'{D}/out/rav_ft_parts'; os.makedirs(P, exist_ok=True)
for f in sorted(glob.glob(f'{OUT}/*')):
    b = open(f,'rb').read(); n = os.path.basename(f)
    for i in range(0, len(b), 9000000):
        open(f'{P}/{n}.part{i // 9000000}','wb').write(b[i:i + 9000000])
print(len(os.listdir(P)), 'parts:', sorted(os.listdir(P))[:5], '...')